In [14]:
# ================================
# F1 Tire Compound Predictiveness Pipeline (Qualifying Version — Fixed)
# ================================

!pip install fastf1 scikit-learn --quiet

import fastf1
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, accuracy_score
import os

# ---------- SETUP ----------
os.makedirs('./f1_cache', exist_ok=True)
fastf1.Cache.enable_cache('./f1_cache')

YEARS = [2022, 2023]
RACES = ['Bahrain', 'Monaco', 'Silverstone', 'Hungary', 'Monza']
all_features = []

# ---------- DATA EXTRACTION ----------
for year in YEARS:
    for race in RACES:
        print(f"\nLoading {race} {year} (Qualifying)...")
        try:
            session = fastf1.get_session(year, race, 'Q')
            session.load()
        except Exception as e:
            print(f"  Session load failed: {e}")
            continue

        laps = session.laps[['Driver', 'LapNumber', 'Compound', 'TyreLife', 'LapTime', 'IsAccurate']].copy()
        laps = laps[laps['IsAccurate'] & laps['LapTime'].notna()]
        if laps.empty:
            print("  No valid laps after filtering. Skipping.")
            continue

        # Convert lap times to seconds
        laps = laps.copy()
        laps.loc[:, 'LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

        # Pull telemetry per driver (aggregated for whole session)
        for drv in laps['Driver'].unique():
            try:
                tel = session.laps.pick_driver(drv).pick_accurate().get_car_data().add_distance()
                avg_speed = tel['Speed'].mean()
                avg_throttle = tel['Throttle'].mean()
                avg_brake = tel['Brake'].mean()

                laps_driver = laps[laps['Driver'] == drv].copy()
                for _, lap in laps_driver.iterrows():
                    all_features.append({
                        'Year': year,
                        'Race': race,
                        'Driver': drv,
                        'LapNumber': lap['LapNumber'],
                        'Compound': lap['Compound'],
                        'TyreLife': lap['TyreLife'],
                        'LapTimeSeconds': lap['LapTimeSeconds'],
                        'AvgSpeed': avg_speed,
                        'AvgThrottle': avg_throttle,
                        'AvgBrake': avg_brake
                    })
            except Exception as e:
                print(f"  Telemetry failed for driver {drv}: {e}")
                continue

        print(f"  Added {len(laps)} laps.")

# ---------- COMBINE ----------
df = pd.DataFrame(all_features)

if df.empty:
    print("\n⚠️  No data collected — likely no telemetry available for these sessions.")
else:
    print(f"\n✅  Collected {len(df)} laps from {df['Race'].nunique()} races.")

    # ---------- FEATURE PROCESSING ----------
    df = df.dropna(subset=['AvgSpeed', 'AvgThrottle', 'AvgBrake', 'TyreLife'])
    df['TyreLife'] = df['TyreLife'].fillna(df['TyreLife'].median())

    # One-hot encode tire compounds
    encoder = OneHotEncoder(drop='first', sparse_output=False)
    compounds_encoded = encoder.fit_transform(df[['Compound']])
    compound_df = pd.DataFrame(compounds_encoded, columns=encoder.get_feature_names_out(['Compound']))
    X_base = df[['AvgSpeed', 'AvgThrottle', 'AvgBrake', 'TyreLife']].reset_index(drop=True)
    X = pd.concat([X_base, compound_df], axis=1)
    y = df['LapTimeSeconds'].reset_index(drop=True)  # lap time is our regression target here

    # ---------- MODEL 1: Linear Regression ----------
    model = LinearRegression().fit(X, y)
    r2 = r2_score(y, model.predict(X))
    coeffs = pd.Series(model.coef_, index=X.columns)

    # Compare with model without tire compound
    X_no_compound = X_base
    model_no = LinearRegression().fit(X_no_compound, y)
    r2_no = r2_score(y, model_no.predict(X_no_compound))

    print("\n=== LINEAR REGRESSION RESULTS ===")
    print(f"R² with compound: {r2:.4f}")
    print(f"R² without compound: {r2_no:.4f}")
    print(f"ΔR² = {r2 - r2_no:.4f}")
    print("\nTop coefficients (with compound):")
    print(coeffs.sort_values(ascending=False).head(10))

    # ---------- MODEL 2: Logistic Classification (Faster vs. Slower Lap) ----------
    # classify laps below driver's median lap time as "faster"
    df['FasterLap'] = (df['LapTimeSeconds'] < df.groupby('Driver')['LapTimeSeconds'].transform('median')).astype(int)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X, df['FasterLap'])
    acc = accuracy_score(df['FasterLap'], clf.predict(X))

    clf_no = LogisticRegression(max_iter=1000)
    clf_no.fit(X_no_compound, df['FasterLap'])
    acc_no = accuracy_score(df['FasterLap'], clf_no.predict(X_no_compound))

    print("\n=== LOGISTIC CLASSIFIER RESULTS ===")
    print(f"Accuracy with compound: {acc:.4f}")
    print(f"Accuracy without compound: {acc_no:.4f}")
    print(f"ΔAccuracy = {acc - acc_no:.4f}")

    # ---------- SUMMARY ----------
    print("\nSummary:")
    if (r2 - r2_no) > 0.01 or (acc - acc_no) > 0.01:
        print("✅ Tire compound shows some predictive power across qualifying sessions.")
    else:
        print("⚠️ Tire compound adds little predictive signal — driver and setup dominate performance.")


core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...



Loading Bahrain 2022 (Qualifying)...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '1', '55', '11', '44', '77', '20', '14', '63', '10', '31', '47', '4', '23', '24', '22', '27', '3', '18', '6']
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be 

  Added 88 laps.

Loading Monaco 2022 (Qualifying)...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '11', '1', '4', '63', '14', '44', '5', '31', '22', '77', '20', '3', '47', '23', '10', '18', '6', '24']
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be r

...

req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '1', '4', '81', '24', '16', '77', '14', '11', '27', '55', '31', '3', '18', '10', '23', '22', '63', '20', '2']
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be 

  Added 141 laps.

Loading Monza 2023 (Qualifying)...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['55', '1', '16', '63', '11', '23', '81', '44', '4', '14', '22', '40', '27', '77', '2', '24', '10', '31', '20', '18']


  Added 135 laps.

✅  Collected 1590 laps from 5 races.

=== LINEAR REGRESSION RESULTS ===
R² with compound: 0.2239
R² without compound: 0.1797
ΔR² = 0.0442

Top coefficients (with compound):
Compound_INTERMEDIATE    12.722874
TyreLife                  0.926148
Compound_SOFT             0.438437
AvgThrottle               0.299769
AvgSpeed                  0.014012
Compound_MEDIUM          -5.810756
AvgBrake                -33.646796
dtype: float64

=== LOGISTIC CLASSIFIER RESULTS ===
Accuracy with compound: 0.6692
Accuracy without compound: 0.6465
ΔAccuracy = 0.0226

Summary:
✅ Tire compound shows some predictive power across qualifying sessions.


/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
/opt/anaconda3/lib/python3.13/site-packages/fast

In [15]:
YEARS = [2022, 2023]
RACES = ['Bahrain', 'Monaco', 'Silverstone', 'Hungary', 'Monza']
all_features = []

for year in YEARS:
    for race in RACES:
        print(f"\n🏁 Loading {race} {year} (Race)...")
        try:
            session = fastf1.get_session(year, race, 'R')
            session.load()
        except Exception as e:
            print(f"  Failed to load race: {e}")
            continue

        laps = session.laps[['Driver', 'LapNumber', 'Compound', 'TyreLife',
                             'LapTime', 'Position', 'IsAccurate']].copy()
        laps = laps[laps['IsAccurate'] & laps['LapTime'].notna()]
        if laps.empty:
            print("  ⚠️ No valid laps after filtering. Skipping.")
            continue

        laps.loc[:, 'LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
        laps.loc[:, 'Position'] = pd.to_numeric(laps['Position'], errors='coerce')
        laps.loc[:, 'PositionChange'] = laps.groupby('Driver')['Position'].diff(-1).fillna(0)

        collected = 0
        for drv in laps['Driver'].unique():
            try:
                tel = session.laps.pick_driver(drv).pick_accurate().get_car_data().add_distance()
                if tel.empty:
                    continue
                avg_speed = tel['Speed'].mean()
                avg_throttle = tel['Throttle'].mean()
                avg_brake = tel['Brake'].mean()

                laps_driver = laps[laps['Driver'] == drv].copy()
                for _, lap in laps_driver.iterrows():
                    all_features.append({
                        'Year': year,
                        'Race': race,
                        'Driver': drv,
                        'LapNumber': lap['LapNumber'],
                        'Compound': lap['Compound'],
                        'TyreLife': lap['TyreLife'],
                        'Position': lap['Position'],
                        'PositionChange': lap['PositionChange'],
                        'AvgSpeed': avg_speed,
                        'AvgThrottle': avg_throttle,
                        'AvgBrake': avg_brake
                    })
                    collected += 1
            except Exception:
                continue

        print(f" Added {collected} laps from {race} {year}")

# ---------- COMBINE ----------
df = pd.DataFrame(all_features)

if df.empty:
    print("\n  No race telemetry available for these sessions (check cache or try FP2).")
else:
    print(f"\n  Collected {len(df)} laps from {df['Race'].nunique()} races.")

    # ---------- FEATURE PROCESSING ----------
    df = df.dropna(subset=['AvgSpeed', 'AvgThrottle', 'AvgBrake', 'TyreLife'])
    df['TyreLife'] = df['TyreLife'].fillna(df['TyreLife'].median())

    encoder = OneHotEncoder(drop='first', sparse_output=False)
    compounds_encoded = encoder.fit_transform(df[['Compound']])
    compound_df = pd.DataFrame(compounds_encoded, columns=encoder.get_feature_names_out(['Compound']))
    X_base = df[['AvgSpeed', 'AvgThrottle', 'AvgBrake', 'TyreLife']].reset_index(drop=True)
    X = pd.concat([X_base, compound_df], axis=1)
    y = df['PositionChange'].reset_index(drop=True)

    # ---------- MODEL 1: Linear Regression ----------
    model = LinearRegression().fit(X, y)
    r2 = r2_score(y, model.predict(X))
    coeffs = pd.Series(model.coef_, index=X.columns)

    X_no_compound = X_base
    model_no = LinearRegression().fit(X_no_compound, y)
    r2_no = r2_score(y, model_no.predict(X_no_compound))

    print("\n=== LINEAR REGRESSION RESULTS ===")
    print(f"R² with compound: {r2:.4f}")
    print(f"R² without compound: {r2_no:.4f}")
    print(f"ΔR² = {r2 - r2_no:.4f}")
    print("\nTop coefficients (with compound):")
    print(coeffs.sort_values(ascending=False).head(10))

    # ---------- MODEL 2: Logistic Classification (Gained vs Lost Position) ----------
    df['GainedPosition'] = (df['PositionChange'] < 0).astype(int)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X, df['GainedPosition'])
    acc = accuracy_score(df['GainedPosition'], clf.predict(X))

    clf_no = LogisticRegression(max_iter=1000)
    clf_no.fit(X_no_compound, df['GainedPosition'])
    acc_no = accuracy_score(df['GainedPosition'], clf_no.predict(X_no_compound))

    print("\n=== LOGISTIC CLASSIFIER RESULTS ===")
    print(f"Accuracy with compound: {acc:.4f}")
    print(f"Accuracy without compound: {acc_no:.4f}")
    print(f"ΔAccuracy = {acc - acc_no:.4f}")

    print("\nSummary:")
    if (r2 - r2_no) > 0.01 or (acc - acc_no) > 0.01:
        print(" Tire compound shows some predictive power across race sessions (position change).")
    else:
        print(" Tire compound adds little predictive signal — race pace and events dominate position changes.")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...



🏁 Loading Bahrain 2022 (Race)...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']
/var/folders/tv/v44ss2l91wd5b6b1sy494b340000gn/T/ipykernel_18896/4167496465.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laps.loc[:, 'LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
/var/folders/tv/v44ss2l91wd5b6b1sy494b3400

  📊 Added 893 laps from Bahrain 2022

🏁 Loading Monaco 2022 (Race)...


core        WARNING 	Fixed incorrect tyre stint information for driver '14'
core        WARNING 	Fixed incorrect tyre stint information for driver '44'
core        WARNING 	Fixed incorrect tyre stint information for driver '77'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Fixed incorrect tyre stint information for driver '10'
core        WARNING 	Fixed incorrect tyre stint information for driver '31'
core        WARNING 	Fixed incorrect tyre stint information for driver '3'
core        WARNING 	Fixed incorrect tyre stint information for driver '18'
core        WARNING 	Fixed incorrect tyre stint information for driver '6'
core        WARNING 	Fixed incorrect tyre stint information for driver '24'
core        WARNING 	Fixed incorrect tyre stint information for driver '22'
core        WARNING 	Fixed incorrect tyre stint information for driver '23'
core        WARNING 	Fixed incorrect tyre stint information for driver '47'
core        WAR

...

  📊 Added 802 laps from Silverstone 2023

🏁 Loading Hungary 2023 (Race)...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '11', '44', '81', '63', '16', '55', '14', '18', '23', '77', '3', '27', '22', '24', '20', '2', '31', '10']
/var/folders/tv/v44ss2l91wd5b6b1sy494b340000gn/T/ipykernel_18896/4167496465.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laps.loc[:, 'LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
/var/folders/tv/v44ss2l91wd5b6b1sy494b340000gn/T/ipykernel_18896/4167496465.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli

  📊 Added 1158 laps from Hungary 2023

🏁 Loading Monza 2023 (Race)...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 06:25.888000 before the recorded end of the session.
core        WARNING 	Driver 11 completed the race distance 06:19.824000 before the recorded end of the session.
core        WARNING 	Driver 55 completed the race distance 06:14.695000 before the recorded end of the session.
core        WARNING 	Driver 16 completed the race distance 06:14.511000 before the recorded end of the session.
core        WARNING 	Driver 63 completed the race distance 06:07.860000 before the recorded end of the session.
core        WARNING 	Driver 44 completed the race distance 05:48.209000 before the recorded end of the session.
core        WARNING 	Driver 23 completed the race distance 05:40.782000 before the recorded end of 

  📊 Added 878 laps from Monza 2023

✅  Collected 9690 laps from 5 races.

=== LINEAR REGRESSION RESULTS ===
R² with compound: 0.0159
R² without compound: 0.0109
ΔR² = 0.0050

Top coefficients (with compound):
AvgBrake                 0.003888
AvgSpeed                 0.000645
AvgThrottle             -0.003265
TyreLife                -0.011535
Compound_None           -0.013274
Compound_INTERMEDIATE   -0.086527
Compound_MEDIUM         -0.089049
Compound_SOFT           -0.149251
Compound_WET            -0.244262
dtype: float64

=== LOGISTIC CLASSIFIER RESULTS ===
Accuracy with compound: 0.9500
Accuracy without compound: 0.9500
ΔAccuracy = 0.0000

Summary:
⚠️ Tire compound adds little predictive signal — race pace and events dominate position changes.
